# HanuAi ML Assessment — Task 1: BestBuy Review Scraping & Sentiment

This notebook is a **policy-respecting starter**. Before running it, check Best Buy Canada's current Terms of Use, robots.txt, and page behavior. Only access pages and fields that are publicly viewable and permitted. If automated access is disallowed, use an approved API/export or a permitted public review dataset instead. Do not bypass CAPTCHA, access controls, or blocks.

## 1. Install dependencies
Run once in your notebook environment. Selenium requires a compatible browser driver; Selenium Manager may resolve it automatically in supported environments.

In [ ]:
%pip install pandas beautifulsoup4 selenium vaderSentiment openpyxl

## 2. Configuration and product selection
Set `PRODUCT_URL` to a BestBuy Canada product page that publicly displays at least 50 reviews. Do not guess selectors; inspect the permitted page structure and update the selectors below. The script deliberately stops if it detects access-denial or CAPTCHA signals.

In [ ]:
import re, time, hashlib, random
from datetime import datetime
from urllib.parse import urlparse
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

PRODUCT_URL = "PASTE_PUBLIC_BESTBUY_CA_PRODUCT_URL_HERE"
SOURCE = "BestBuy Canada"
MAX_CLICKS_PER_SORT = 30
SORT_OPTIONS = ["Most Helpful", "Newest", "Highest Rating", "Lowest Rating", "Most Relevant"]

# Use only normal browser settings and modest pacing. No proxy rotation or CAPTCHA bypass.
options = webdriver.ChromeOptions()
options.add_argument("--window-size=1365,900")
# Keep the browser visible so the operator can see access challenges and stop if needed.
driver = webdriver.Chrome(options=options)

## 3. Safe navigation, retry and review extraction
Selectors vary by page implementation, so verify them in browser devtools. `Show More` is attempted only when present. Sorting options are attempted by visible text; unavailable options are skipped. A page that presents CAPTCHA/access denial should be stopped rather than circumvented.

In [ ]:
def navigate_with_retry(url, attempts=3):
    for attempt in range(attempts):
        try:
            driver.get(url)
            time.sleep(2 + random.uniform(0.5, 1.5))  # gentle pacing
            page = driver.page_source.lower()
            if any(s in page for s in ["verify you are human", "captcha", "access denied"]):
                raise RuntimeError("Access challenge detected. Stop and use an approved access method.")
            return
        except RuntimeError:
            raise
        except WebDriverException:
            if attempt == attempts - 1: raise
            time.sleep(2 ** attempt)  # exponential backoff for transient failures

def click_visible_text(text):
    # Generic fallback; inspect the page and replace with a stable, site-approved selector if needed.
    candidates = driver.find_elements(By.XPATH, f"//*[self::button or self::a or @role='button'][contains(translate(normalize-space(.),'ABCDEFGHIJKLMNOPQRSTUVWXYZ','abcdefghijklmnopqrstuvwxyz'), '{text.lower()}')]")
    for el in candidates:
        try:
            if el.is_displayed() and el.is_enabled():
                driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
                el.click()
                time.sleep(1.5)
                return True
        except WebDriverException:
            continue
    return False

def scrape_visible_reviews(sort_label):
    records = []
    for _ in range(MAX_CLICKS_PER_SORT):
        soup = BeautifulSoup(driver.page_source, "html.parser")
        # IMPORTANT: confirm these selectors against the current public page before use.
        review_nodes = soup.select('[data-testid*="review"], .review-item, .review')
        for node in review_nodes:
            text = lambda selectors: next((node.select_one(s).get_text(" ", strip=True)
                                           for s in selectors if node.select_one(s)), "")
            title = text(['[data-testid*="title"]', '.review-title', 'h3'])
            body = text(['[data-testid*="body"]', '.review-text', '.review-content', 'p'])
            name = text(['[data-testid*="author"]', '.review-author', '.author'])
            rating_text = text(['[aria-label*="out of 5"]', '[data-testid*="rating"]', '.rating'])
            date_text = text(['time', '[data-testid*="date"]', '.review-date'])
            rating_match = re.search(r'([1-5](?:\.0)?)\s*(?:out of|/\s*)?\s*5?', rating_text, re.I)
            rating = float(rating_match.group(1)) if rating_match else None
            date_value = pd.to_datetime(date_text, errors="coerce")
            date_value = date_value.strftime("%Y-%m-%d") if pd.notna(date_value) else ""
            if title or body:
                raw = "|".join([title, body, date_value, str(rating), name])
                records.append({"Title": title, "Review Text": body, "Date": date_value,
                                "Rating": rating, "Source": SOURCE, "Reviewer Name": name,
                                "Sort Filter": sort_label,
                                "Primary Key": hashlib.sha256(raw.encode("utf-8")).hexdigest()[:16]})
        if not click_visible_text("Show More"):
            break
    # De-duplicate records across overlapping sort views
    return pd.DataFrame(records).drop_duplicates(subset=["Primary Key"])

if PRODUCT_URL.startswith("http"):
    navigate_with_retry(PRODUCT_URL)
else:
    raise ValueError("Set PRODUCT_URL to a valid public product URL before running.")

## 4. Apply sort options and combine results
Sorting controls are not guaranteed to exist or be accessible on every product page. Confirm that the selected sort actually changed; never label records with a filter that was not applied.

In [ ]:
all_reviews = []
for sort_label in SORT_OPTIONS:
    # Re-load the product page between sort passes.
    navigate_with_retry(PRODUCT_URL)
    # Open reviews section if needed.
    click_visible_text("Customer Reviews")
    # Try the requested sort label. If the control is not found, skip this pass.
    applied = click_visible_text(sort_label)
    if not applied and sort_label != "Most Relevant":
        print("Sort option unavailable; skipping:", sort_label)
        continue
    chunk = scrape_visible_reviews(sort_label)
    if not chunk.empty:
        all_reviews.append(chunk)
    time.sleep(2 + random.uniform(0.5, 1.5))

reviews = pd.concat(all_reviews, ignore_index=True) if all_reviews else pd.DataFrame(
    columns=["Primary Key","Title","Review Text","Date","Rating","Source","Reviewer Name","Sort Filter"])
reviews = reviews.drop_duplicates(subset=["Primary Key"])
print("Unique public reviews collected:", len(reviews))
reviews.to_csv("bestbuy_reviews_raw.csv", index=False, encoding="utf-8-sig")

## 5. Sentiment and functional aspect tags
VADER is a lightweight baseline for English text; it may misread sarcasm, mixed sentiment, short text and non-English reviews. A transformer model can be substituted after evaluating it on manually labelled examples.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
analyzer = SentimentIntensityAnalyzer()

def classify_sentiment(text):
    score = analyzer.polarity_scores(str(text))["compound"]
    if score >= 0.05: label = "Positive"
    elif score <= -0.05: label = "Negative"
    else: label = "Neutral"
    return pd.Series([label, score])

reviews["Combined Text"] = (reviews["Title"].fillna("") + ". " +
                            reviews["Review Text"].fillna("")).str.strip()
reviews[["Sentiment","Sentiment Score"]] = reviews["Combined Text"].apply(classify_sentiment)

aspect_rules = {
    "Design & Quality": [r"design", r"quality", r"look", r"build"],
    "Performance": [r"work", r"performance", r"fast", r"slow", r"power"],
    "Ease of Use / Setup": [r"easy", r"setup", r"install", r"assemble"],
    "Value / Price": [r"price", r"value", r"cost", r"expensive"],
    "Reliability / Defect": [r"broken", r"fail", r"defect", r"stopped", r"return"],
    "Delivery / Service": [r"delivery", r"shipping", r"service", r"support"]
}
def aspect_sentiment(text, sentiment):
    t = str(text).lower()
    tags = [aspect for aspect, patterns in aspect_rules.items()
            if any(re.search(p, t) for p in patterns)]
    if not tags: tags = ["General Experience"]
    return [f"{tag} ({'Pos' if sentiment=='Positive' else 'Neg' if sentiment=='Negative' else 'Neu'})"
            for tag in tags]
reviews["Sentiment Categories"] = reviews.apply(
    lambda r: aspect_sentiment(r["Combined Text"], r["Sentiment"]), axis=1)
reviews.to_csv("bestbuy_reviews_with_sentiment.csv", index=False, encoding="utf-8-sig")
reviews.to_excel("bestbuy_reviews_with_sentiment.xlsx", index=False)
reviews.head()

## 6. Responsible scraping challenge handling
- **Rate limits / transient failures:** use low request frequency, exponential backoff, bounded retries, and stop on repeated errors.
- **IP blocking / CAPTCHA / bot checks:** do not evade or bypass them. Stop automated collection, review the site's terms and robots policy, and request permission or use an approved API/data export.
- **User-Agent / headers:** do not impersonate another client to conceal automation. Use standard browser behavior and identify the project if required by site policy.
- **Proxy rotation:** only consider approved organizational proxies for legitimate network reliability, never to evade a site's access restrictions or rate limits.
- **Pagination and filters:** verify that a control exists and is applied before collecting data; deduplicate reviews across sort views by stable content hash.
- **Data minimization:** collect only fields needed for the assignment, avoid sensitive data, and do not republish reviewer details beyond the assignment's legitimate scope.

## 7. Stakeholder summary template
After successful collection, calculate rating and sentiment distributions, review aspect tags, and compare low-star reviews against recurring complaint themes. Do not report fabricated results if the site prevents permitted access. Treat automated sentiment as a prioritization signal and validate a sample manually.